# Benchmark models

In [2]:
from pathlib import Path, PurePosixPath
from urllib.parse import urlparse, parse_qs
import json
import os
import shutil
import yaml

## Creating the Validation Dataset

In [7]:
ROOT = Path.cwd().resolve()
if not (ROOT / "configs/classes.yaml").exists():
    ROOT = ROOT.parent

ANNOTATIONS = ROOT / "data/annotations/label_studio/20260728T210000_poco-f3/annotations.json"
RAW_ROOT = ROOT / "data/raw"
BENCHMARK = ROOT / "data/processed/benchmarks/20260728T210000_poco-f3"
IMAGES = BENCHMARK / "images"
LABELS = BENCHMARK / "labels"

IMAGES.mkdir(parents=True, exist_ok=True)
LABELS.mkdir(parents=True, exist_ok=True)

class_config = yaml.safe_load((ROOT / "configs/classes.yaml").read_text())
names = {int(k): v for k, v in class_config["names"].items()}
class_ids = {name: class_id for class_id, name in names.items()}
tasks = json.loads(ANNOTATIONS.read_text())

image_count = 0
box_count = 0

for task in tasks:
    annotations = [
        a for a in task.get("annotations", [])
        if not a.get("was_cancelled")
    ]
    if not annotations:
        continue

    annotation = max(annotations, key=lambda a: a.get("updated_at", ""))
    relative = parse_qs(urlparse(task["data"]["image"]).query)["d"][0]
    source = RAW_ROOT / Path(*PurePosixPath(relative).parts)

    destination = IMAGES / source.name
    if not destination.exists():
        try:
            os.link(source, destination)  # NTFS hardlink: no duplicated image data
        except OSError:
            shutil.copy2(source, destination)

    lines = []
    for result in annotation.get("result", []):
        if result.get("type") != "rectanglelabels":
            continue

        value = result["value"]
        label = value["rectanglelabels"][0]
        class_id = class_ids[label]

        x = value["x"] / 100
        y = value["y"] / 100
        width = value["width"] / 100
        height = value["height"] / 100
        x_center = x + width / 2
        y_center = y + height / 2

        lines.append(
            f"{class_id} {x_center:.8f} {y_center:.8f} "
            f"{width:.8f} {height:.8f}"
        )
        box_count += 1

    (LABELS / f"{source.stem}.txt").write_text(
        "\n".join(lines) + ("\n" if lines else "")
    )
    image_count += 1

dataset_yaml = BENCHMARK / "dataset.yaml"
dataset_yaml.write_text(yaml.safe_dump({
    "path": BENCHMARK.as_posix(),
    "train": "images",  # required by Ultralytics but unused by model.val()
    "val": "images",
    "names": names,
}, sort_keys=False))

print("Images:", image_count)
print("Boxes:", box_count)
print("Dataset:", dataset_yaml)

Images: 97
Boxes: 1029
Dataset: C:\Users\maxan\Hobby\Inforelektrikbau\Tichu-Bot\software\tichu-card-detection\data\processed\benchmarks\20260728T210000_poco-f3\dataset.yaml


## Benchmarking the models

In [ ]:
from ultralytics import YOLO
import pandas as pd

models = {
    "color": ROOT / "models/trained/tichu_yolo26m/weights/best.pt",
    "color_finetuned": ROOT / "models/trained/tichu_yolo26m2/weights/best.pt",
    "mixed": ROOT / "models/trained/tichu_yolo26m_mix/weights/best.pt",
    "monochrome": ROOT / "models/trained/tichu_yolo26m_monochrome/weights/best.pt",
}

report_dir = ROOT / "reports/benchmarks/20260728T210000_poco-f3"
rows = []

for run_name, weights in models.items():
    model = YOLO(str(weights))
    model.model.names = names  # display canonical names

    metrics = model.val(
        data=str(dataset_yaml),
        split="val",
        imgsz=960,
        conf=0.001,       # needed for reliable PR and AP curves
        iou=0.7,          # NMS threshold
        max_det=300,
        batch=8,
        workers=0,        # safer inside Windows notebooks
        device=0,
        plots=True,
        project=str(report_dir),
        name=run_name,
        exist_ok=True,
    )

    rows.append({
        "model": run_name,
        **metrics.results_dict,
    })

summary = pd.DataFrame(rows)
summary.to_csv(report_dir / "summary.csv", index=False)
summary

Ultralytics 8.4.108  Python-3.12.2 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3080, 10240MiB)
YOLO26m summary (fused): 132 layers, 20,392,628 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1759.5241.1 MB/s, size: 343.8 KB)
val: Scanning C:\Users\maxan\Hobby\Inforelektrikbau\Tichu-Bot\software\tichu-card-detection\data\processed\benchmarks\20260728T210000_poco-f3\labels... 97 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 97/97 923.8it/s 0.1s
val: C:\Users\maxan\Hobby\Inforelektrikbau\Tichu-Bot\software\tichu-card-detection\data\processed\benchmarks\20260728T210000_poco-f3\images\frame_000122.jpg: 1 duplicate labels removed
val: New cache created: C:\Users\maxan\Hobby\Inforelektrikbau\Tichu-Bot\software\tichu-card-detection\data\processed\benchmarks\20260728T210000_poco-f3\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 4.9it/s 2.7s0.2s
                   all     

,model,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),fitness
0,color,0.912686,0.850112,0.899588,0.853612,0.853612
1,color_finetuned,0.923315,0.804554,0.878335,0.823419,0.823419
2,mixed,0.899029,0.869885,0.906991,0.888634,0.888634
3,monochrome,0.847741,0.724981,0.829099,0.778361,0.778361
